In [1]:
from ipywidgets import interact, FloatSlider
from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np

Insert here the crimper dimansions

In [2]:
N = 12

Rp = 37.7

drive_ring_travel = 20


#crimp head geometry

bd = 11.3
sd = -0.8

bp =37.6
sp = -2.6

drive_pin_diameter = 3

W = 0.16*Rp
bl = W / np.tan((2*np.pi/N))
lb = max(bp + 0.09*Rp, bl + 0.08*Rp)

print(W)



6.032000000000001


In [3]:
print(f"blade angle= {np.rad2deg(2*np.pi/N)}")

a = bp-bd

print(a)
c = sd - sp
print(c)

L = np.hypot(a,c)
print(L)

max_hub_angle = np.asin(L/Rp)

print(np.rad2deg(max_hub_angle))

blade angle= 29.999999999999996
26.3
1.8
26.36152499382386
44.36643712647596


In [4]:
def crimper_mechanism(drive_ring_angle):
    O=[]
    P=[]
    Q=[]
    w=[]
    u=[]
    n=[]
    T=[]
    Rb=[]
    body=[]

    for i in range(N):
        alpha = (2*np.pi/(N))*i

        O.append(np.array([0,0]))
        P.append(np.array([Rp*np.cos(alpha),Rp*np.sin(alpha)]))
    
        delta = np.deg2rad(drive_ring_angle)
        # calculate the distace to the drive ring for a given drive ring angle
        r = Rp*np.cos(delta) - np.sqrt(L**2 - (Rp*np.sin(delta))**2)
    
        # Q is the location of the drive pin
    
        Q.append( r*np.array([np.cos(delta+alpha), np.sin(delta+alpha)]))
        
        w.append(Q[i] - P[i])
    
        # orientation of the crimper head
    
        beta = np.atan2(w[i][1],w[i][0])
    
        
        phi = beta - np.atan2(c,a)
    
        # cordinate system u and n in global cordinates
        
        u.append(np.array([np.cos(phi), np.sin(phi)]))
        n.append(np.array([-np.sin(phi), np.cos(phi)]))
    
        # location of crimp head tip 
        
        T.append(Q[i] + bd*u[i] - sd*n[i])
       # assert np.allclose(T[i], P[i] + bp*u[i] - sp*n[i]) 

        # body of the crimper head
        
        # blade coordinates (b behind the tip, s sideways) of the four corners
        local = np.array([[0,  0],     # tip T
                          [bl, -W],    # end of the angled face
                          [lb, -W],    # rear, far side
                          [lb,  0],
                          [0,0]])
        # rear, on the flat face line
        
        Rb.append(np.column_stack((-u[i], n[i]))) # column 1 = -u (backwards), column 2 = n (sideways)
        body.append(T[i] + local @ Rb[i].T )       # one row per corner, in world coordinates
    print(f"D_tip = {2*np.hypot(T[0][0],T[0][1])}")
        
        

    return O,Q,P,T,body

In [9]:
def plot_mechanism(drive_angle):

    O,Q,P,T,body = crimper_mechanism(drive_angle)

    fig, ax = plt.subplots(figsize=(7, 6))

    print(np.hypot(Q[0][0],Q[0][1]))

    for i in range(N):

        ax.plot(
            [O[i][0], Q[i][0]],
            [O[i][1], Q[i][1]],
            'g--',
            linewidth=0,
            label='r'
        )
    
        ax.plot(
            [P[i][0], Q[i][0]],
            [P[i][1], Q[i][1]],
            'g--',
            linewidth=0,
            label='L'
        )
    
    
        x = []
        y = []
        for point in body[i]:
            x.append(point[0])
            y.append(point[1])
    
        ax.plot(
            x,
            y,
            'k-',
            linewidth=1,
            label='T'
        )

    

    
    
    ax.set_aspect('equal')
    ax.grid(True)

    # Center view around origin
    ax.set_xlim(-50, 50)
    ax.set_ylim(-50, 50)
    
    # Put axes through (0,0)


    ax.set_title(f"Actuation angle = {drive_angle:.1f}°")

    plt.show()


In [10]:
interact(
    plot_mechanism,
    drive_angle=FloatSlider(
        min=0,
        max=np.rad2deg(max_hub_angle)-1,
        step=1,
        value=20,
        description='Angle'
    )
);

interactive(children=(FloatSlider(value=20.0, description='Angle', max=43.36643712647596, step=1.0), Output())…